# 05 Â· Push to Hub â€” Publish Your Adapter

**Runs on: Local CPU (or Colab)**  
**Prerequisite**: Adapter at `outputs/adapter/` from notebook 03.

This notebook pushes your LoRA adapter to the HF Hub. Because the adapter is only ~5 MB,  
anyone can load your model with just one extra line: `PeftModel.from_pretrained(base, 'your-username/repo')`.

In [ ]:
# !pip install huggingface_hub peft transformers

## 1. Login to Hugging Face

In [ ]:
import os
from huggingface_hub import login, whoami

token = os.environ.get("HF_TOKEN_NEW")
if not token:
    raise EnvironmentError("Set the HF_TOKEN_NEW environment variable to your Hugging Face token")

login(token=token)
me = whoami()
HF_USERNAME = me["name"]
print(f"Logged in as: {HF_USERNAME} {token}")


## 2. Configure the Repository Name

In [ ]:
ADAPTER_PATH = "../outputs/adapter"
REPO_NAME = f"{HF_USERNAME}/smollm2-135m-smoltalk-lora"

print(f"Will push to: https://huggingface.co/{REPO_NAME}")

## 3. Write a Model Card

A model card (`README.md` on the Hub) documents what the model does, how it was trained, and how to use it.

In [ ]:
import os

model_card = f"""---
base_model: HuggingFaceTB/SmolLM2-135M
library_name: peft
tags:
  - lora
  - instruction-tuning
  - smoltalk
  - learning-project
---

# SmolLM2-135M â€” Instruction Fine-Tune (LoRA)

A LoRA adapter for [HuggingFaceTB/SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M)  
fine-tuned on 2,000 samples from [HuggingFaceTB/smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) (everyday-conversations config).

This is a **learning project** â€” not a production model. It demonstrates that even 2k samples of  
LoRA SFT teaches a base model the instruction-following format.

## Training Details

| | |
|---|---|
| Base model | SmolLM2-135M |
| Dataset | smoltalk/everyday-conversations (2k samples) |
| Epochs | 1 |
| LoRA r | 8 |
| LoRA alpha | 16 |
| Target modules | q_proj, v_proj |
| max_seq_length | 1024 |
| Compute | Google Colab T4 (~15 min) |

## How to Use

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_ID = "HuggingFaceTB/SmolLM2-135M"
ADAPTER_ID = "{REPO_NAME}"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)
base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.float32)
model = PeftModel.from_pretrained(base, ADAPTER_ID)
model.eval()

messages = [{{"role": "user", "content": "What is gravity?"}}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.3,
                         pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
```
"""

card_path = os.path.join(ADAPTER_PATH, "README.md")
with open(card_path, "w") as f:
    f.write(model_card)
print(f"Model card written to {card_path}")

## 4. Push to Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=REPO_NAME,
    repo_type="model",
)
print(f"Done! https://huggingface.co/{REPO_NAME}")


## 5. Verify â€” Load from Hub

This downloads your adapter back from the Hub to confirm it works.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("Loading your model from Hub...")
hub_tok = AutoTokenizer.from_pretrained(REPO_NAME)
hub_base = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M", torch_dtype=torch.float32)
hub_model = PeftModel.from_pretrained(hub_base, REPO_NAME)
hub_model.eval()
print("Loaded successfully!")

messages = [{"role": "user", "content": "What is the largest planet in our solar system?"}]
prompt = hub_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = hub_tok(prompt, return_tensors="pt")

with torch.no_grad():
    out = hub_model.generate(
        **inputs, max_new_tokens=80, do_sample=True, temperature=0.3,
        pad_token_id=hub_tok.eos_token_id
    )
new_toks = out[0][inputs["input_ids"].shape[1]:]
print(f"\nQ: {messages[0]['content']}")
print(f"A: {hub_tok.decode(new_toks, skip_special_tokens=True)}")

## Congratulations!

You've completed the full LLM fine-tuning learning loop:

1. **Inference** â€” understood tokens, logits, generation sampling
2. **Dataset** â€” chat format, chat templates, sequence lengths
3. **Fine-tuning** â€” LoRA, SFTTrainer, training on a T4
4. **Evaluation** â€” qualitative before/after comparison
5. **Deployment** â€” pushed a live model to the Hugging Face Hub

---

**Suggested next experiments:**
- Try a 7B model with QLoRA (4-bit quantization) â€” still fits on a Colab T4
- Try DPO (Direct Preference Optimization) with preference pairs
- Run the `lm-evaluation-harness` to get benchmark numbers
- Merge LoRA into the base model with `model.merge_and_unload()` for a single standalone checkpoint